In [23]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
import numpy as np
import pandas as pd

In [24]:
train=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/train.csv')
test=pd.read_csv('/Users/taha/Downloads/playground-series-s4e6/test.csv')

In [25]:
# Séparation de X et y sur l'intégralité du jeu d'entraînement
X = train.drop(columns=["id", "Target"])
y = train["Target"]

# Encodage de la variable cible sur TOUT le jeu 'y'
le = LabelEncoder()
y_encoded = le.fit_transform(y)
# Ordre alphabétique : Dropout=0, Enrolled=1, Graduate=2

In [26]:
X.nunique()

Marital status                                       6
Application mode                                    22
Application order                                    8
Course                                              19
Daytime/evening attendance                           2
Previous qualification                              21
Previous qualification (grade)                     110
Nacionality                                         18
Mother's qualification                              35
Father's qualification                              39
Mother's occupation                                 40
Father's occupation                                 56
Admission grade                                    668
Displaced                                            2
Educational special needs                            2
Debtor                                               2
Tuition fees up to date                              2
Gender                                               2
Scholarshi

In [27]:
cat_cols = [
    'Marital status', 'Application mode', 'Course', 'Previous qualification', 
    'Nacionality', "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation", 'Displaced', 
    'Educational special needs', 'Debtor', 'Tuition fees up to date', 
    'Gender', 'Scholarship holder', 'International'
]

In [28]:
def feature_engineering(df):
    df_feat = df.copy()
    
    # 1. Ratios de réussite académique
    df_feat['rate_1st_sem'] = df_feat['Curricular units 1st sem (approved)'] / (df_feat['Curricular units 1st sem (enrolled)'] + 1e-5)
    df_feat['rate_2nd_sem'] = df_feat['Curricular units 2nd sem (approved)'] / (df_feat['Curricular units 2nd sem (enrolled)'] + 1e-5)
    
    # 2. Cumuls annuels
    df_feat['total_approved'] = df_feat['Curricular units 1st sem (approved)'] + df_feat['Curricular units 2nd sem (approved)']
    df_feat['total_enrolled'] = df_feat['Curricular units 1st sem (enrolled)'] + df_feat['Curricular units 2nd sem (enrolled)']
    df_feat['overall_approval_rate'] = df_feat['total_approved'] / (df_feat['total_enrolled'] + 1e-5)
    
    # 3. Progression entre semestres
    df_feat['grade_progression'] = df_feat['Curricular units 2nd sem (grade)'] - df_feat['Curricular units 1st sem (grade)']
    df_feat['approved_progression'] = df_feat['Curricular units 2nd sem (approved)'] - df_feat['Curricular units 1st sem (approved)']
    
    # 4. Instabilité financière (Positif = à risque)
    df_feat['financial_instability'] = (df_feat['Debtor'] == 1).astype(int) - (df_feat['Tuition fees up to date'] == 1).astype(int) - (df_feat['Scholarship holder'] == 1).astype(int)
    
    return df_feat

# Application sur X et sur le jeu de test complet
X_engineered = feature_engineering(X)
X_test = test.drop(columns=["id"], errors="ignore")
X_test_engineered = feature_engineering(X_test)

# Conversion des variables catégorielles pour les modèles
for col in cat_cols:
    X_engineered[col] = X_engineered[col].astype('category')
    X_test_engineered[col] = X_test_engineered[col].astype('category')

In [29]:
# 1. Initialisation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Matrices OOF (Out-Of-Fold) pour l'entraînement
lgb_oof_probs = np.zeros((len(X_engineered), 3))
cb_oof_probs = np.zeros((len(X_engineered), 3))
xgb_oof_probs = np.zeros((len(X_engineered), 3))

# Matrices pour cumuler les prédictions du jeu de test
lgb_test_probs = np.zeros((len(X_test_engineered), 3))
cb_test_probs = np.zeros((len(X_test_engineered), 3))
xgb_test_probs = np.zeros((len(X_test_engineered), 3))

print("Début de l'entraînement 5-Fold avec le Trio (CatBoost, LightGBM, XGBoost)...")

# Harmonisation des catégories pour éviter l'erreur XGBoost
for col in cat_cols:
    # 1. On trouve toutes les valeurs uniques possibles en combinant Train et Test
    all_categories = pd.concat([X_engineered[col], X_test_engineered[col]]).unique()
    
    # 2. On applique exactement la même structure catégorielle aux deux DataFrames
    X_engineered[col] = pd.Categorical(X_engineered[col], categories=all_categories)
    X_test_engineered[col] = pd.Categorical(X_test_engineered[col], categories=all_categories)
    
for fold, (train_idx, val_idx) in enumerate(skf.split(X_engineered, y_encoded)):
    print(f"\n--- Pli {fold + 1}/5 ---")
    X_tr, y_tr = X_engineered.iloc[train_idx], y_encoded[train_idx]
    X_va, y_va = X_engineered.iloc[val_idx], y_encoded[val_idx]
    
    # --- 1. LightGBM ---
    lgb_fold = lgb.LGBMClassifier(
        n_estimators=700, learning_rate=0.03, num_leaves=31, max_depth=8,
        subsample=0.8, colsample_bytree=0.8, objective='multiclass', num_class=3,
        random_state=42 + fold, verbosity=-1
    )
    lgb_fold.fit(X_tr, y_tr)
    lgb_oof_probs[val_idx] = lgb_fold.predict_proba(X_va)
    lgb_test_probs += lgb_fold.predict_proba(X_test_engineered) / skf.n_splits
    
    # --- 2. CatBoost ---
    cb_fold = CatBoostClassifier(
        iterations=700, learning_rate=0.03, depth=6, loss_function='MultiClass',
        random_seed=42 + fold, verbose=0, cat_features=cat_cols
    )
    cb_fold.fit(X_tr, y_tr)
    cb_oof_probs[val_idx] = cb_fold.predict_proba(X_va)
    cb_test_probs += cb_fold.predict_proba(X_test_engineered) / skf.n_splits
    
    # --- 3. XGBoost ---
    # XGBoost nécessite tree_method='hist' pour lire les colonnes catégorielles Pandas
    xgb_fold = xgb.XGBClassifier(
        n_estimators=700, learning_rate=0.03, max_depth=6, 
        subsample=0.8, colsample_bytree=0.8, objective='multi:softprob', 
        num_class=3, random_state=42 + fold, 
        enable_categorical=True, tree_method='hist' 
    )
    xgb_fold.fit(X_tr, y_tr)
    xgb_oof_probs[val_idx] = xgb_fold.predict_proba(X_va)
    xgb_test_probs += xgb_fold.predict_proba(X_test_engineered) / skf.n_splits

print("\nEntraînement terminé ! Recherche des poids idéaux pour le Trio...")

# 2. Recherche du Blend optimal à 3 variables
best_acc = 0.0
best_weights = (0, 0, 0)

# On teste des combinaisons par pas de 5% (0.0, 0.05, 0.10, etc.)
for w_cb in np.linspace(0, 1, 21):
    for w_lgb in np.linspace(0, 1 - w_cb, 21):
        w_xgb = 1.0 - w_cb - w_lgb
        # On s'assure que la somme des poids vaut bien 1
        if w_xgb < -1e-5: 
            continue
            
        blended_oof = (w_cb * cb_oof_probs) + (w_lgb * lgb_oof_probs) + (w_xgb * xgb_oof_probs)
        acc = accuracy_score(y_encoded, np.argmax(blended_oof, axis=1))
        
        if acc > best_acc:
            best_acc = acc
            best_weights = (w_cb, w_lgb, w_xgb)

w_cb_final, w_lgb_final, w_xgb_final = best_weights
print(f"Meilleurs Poids -> CatBoost: {w_cb_final:.2f} | LightGBM: {w_lgb_final:.2f} | XGBoost: {w_xgb_final:.2f}")
print(f"Accuracy finale estimée (Blend OOF) : {best_acc:.5f}")

# 3. Génération du Fichier de Soumission Kaggle
final_test_probs = (w_cb_final * cb_test_probs) + (w_lgb_final * lgb_test_probs) + (w_xgb_final * xgb_test_probs)
final_preds_encoded = np.argmax(final_test_probs, axis=1)

# Reconversion en labels textes ('Dropout', etc.)
final_preds_labels = le.inverse_transform(final_preds_encoded)

submission_trio = pd.DataFrame({
    'id': test['id'],
    'Target': final_preds_labels
})

submission_trio.to_csv('submission_trio_xgboost.csv', index=False)
print("\nFichier submission_trio_xgboost.csv généré avec succès ! Prêt pour Kaggle.")

Début de l'entraînement 5-Fold avec le Trio (CatBoost, LightGBM, XGBoost)...

--- Pli 1/5 ---

--- Pli 2/5 ---

--- Pli 3/5 ---

--- Pli 4/5 ---

--- Pli 5/5 ---

Entraînement terminé ! Recherche des poids idéaux pour le Trio...
Meilleurs Poids -> CatBoost: 0.10 | LightGBM: 0.27 | XGBoost: 0.63
Accuracy finale estimée (Blend OOF) : 0.83271

Fichier submission_trio_xgboost.csv généré avec succès ! Prêt pour Kaggle.
